# 06 · Functional programming, iterators, and decorators

Functional techniques help you express a transformation as small functions that are easy
to combine and test. You will compare loops, comprehensions, `map`, and `filter`; follow
lazy iterators; write generators and closures; and build decorators that preserve results.

**Before you begin:** know functions, loops, and collections. A short iterator class is
explained from scratch, so this chapter runs independently of Chapter 05. Run cells from
top to bottom. Every consumed stream is finite or explicitly bounded, and expected errors
are caught. Six practice exercises have separate, self-contained
[solutions](../solutions/06_functional_programming_solutions.ipynb).


## 1. Pure functions and side effects

A **pure function** produces a result determined by its arguments and has no externally
observable side effects. Mutating a caller's list, writing a file, printing, reading input,
or depending on changing global state makes behavior less self-contained. Purity is about
observable behavior: local variables and a local temporary list are compatible with a pure
function. The printing below belongs to the caller, separate from the transformation.

Small transformations are easier to reason about, test, reuse, and compose. Python is a
multi-paradigm language, so functional, procedural, and object-oriented code can coexist.
Even strongly functional languages have names and ways to model effects; they are not
languages "without variables." Python does not enforce purity for you.


In [ ]:
def add_tax(prices, rate):
    """Return new values without mutating the caller's input sequence."""
    return [price * (1 + rate) for price in prices]

prices = [10, 20]
adjusted = add_tax(prices, 0.2)
print(adjusted, prices)
assert prices == [10, 20]
assert adjusted == add_tax(prices, 0.2)

def append_price_in_place(values, price):
    values.append(price)  # A deliberate, observable side effect.

copied_prices = prices.copy()
append_price_in_place(copied_prices, 30)
print(copied_prices)


## 2. Functions are first-class objects

A function can be assigned to a name, stored in a collection, passed as an argument, and
returned as a result. Use `function` to pass the object; `function(...)` calls it now.
A function that accepts or returns a function is often called **higher-order**.

`perform_twice` below forwards positional arguments (`*args`) and keyword arguments
(`**kwargs`). Returning both results avoids silently throwing them away. With a pure
function the two values match; with a stateful function, two calls can produce two effects.


In [ ]:
def perform_twice(function, *args, **kwargs):
    """Call a function twice and preserve both return values."""
    return function(*args, **kwargs), function(*args, **kwargs)

def combine(a, b, separator="-"):
    return f"{a}{separator}{b}"

operation = combine
print(operation is combine)
print(perform_twice(operation, "data", "science", separator=" / "))


## 3. Transform every element with `map`

The repeated pattern "apply a function and collect each result" can be written as a loop,
a comprehension, or `map(function, iterable)`. Pass the function itself as the first
argument. In Python 3, `map` returns a lazy iterator, not a list. `list(...)` consumes that
iterator and stores its results. All three forms preserve input order.

`map` can also receive several iterables. It passes one value from each to the function and
stops when the shortest iterable is exhausted. Choose the form that makes the operation
easiest to read; no form is always the fastest.


In [ ]:
languages = ["python", "perl", "java", "c++"]
lengths_loop = []
for language in languages:
    lengths_loop.append(len(language))
lengths_comprehension = [len(language) for language in languages]
lengths_map = list(map(len, languages))
print(lengths_loop, lengths_comprehension, lengths_map)
print(list(map(float, ["1.0", "3.3", "-4.2"])))
print(list(map(sum, [[1, 3], [4, 2, -5]])))
print(list(map(str, [1, True, [2, 3]])))
print(list(map(pow, [2, 3, 4], [3, 2])))


## 4. Keep matching elements with `filter`

A **predicate** is a function used to decide whether a value should be kept. `filter`
preserves the original values whose predicate result is truthy; it does not replace them
with `True`. Passing `None` as the predicate keeps truthy values. Be careful: a truthy string
such as `"A"` is kept too. To select nonzero integers from mixed data, state both conditions;
`type(value) is int` below intentionally excludes booleans.

The prime-number predicate returns `False` for values below two and tries divisors only
while their square is no greater than the number. This is enough because a composite
number must have at least one factor no greater than its square root.


In [ ]:
def is_prime(number):
    """Return whether an integer is prime; numbers below two are not prime."""
    if number < 2:
        return False
    divisor = 2
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 1
    return True

fibs = [1, 1, 2, 3, 5, 8, 13, 21, 34]
def is_even(number):
    return number % 2 == 0

print(list(filter(is_even, fibs)))
print([number for number in fibs if is_even(number)])
print(list(filter(is_prime, range(30))))
mixed = [0, 1, 0, 6, "A", True, 1, 0, 7]
print(list(filter(None, mixed)))
print(list(filter(lambda value: type(value) is int and value != 0, mixed)))


## 5. Compose transformations

Build a pipeline by making one operation's output the next operation's input. This pipeline
strips whitespace, converts text to lowercase, then removes empty strings. It creates
new strings and leaves the original list unchanged. The same work can be expressed with a
comprehension; choose the form whose steps are clearest for your reader.

Neither `map` nor `filter` guarantees purity: a supplied function may have side effects.
Their functional interface is useful, but the function's behavior still matters.


In [ ]:
def cleaned_words(words):
    """Lazily strip, lowercase and keep nonempty text values."""
    return filter(None, map(lambda word: word.strip().lower(), words))

raw_words = [" Python ", "", "  ", " DATA "]
clean = cleaned_words(raw_words)
print(list(clean))
print(raw_words)
stripped = (word.strip().lower() for word in raw_words)
print([word for word in stripped if word])


## 6. Lazy means the work happens when a value is requested

Creating `map` does not call the transformation for every input. `next` requests one result;
`list` then requests all remaining results. The iterator remembers how far it has progressed.
After it is exhausted, it stays exhausted. Laziness can save computation and avoid storing
all results, but calling `list` still stores the full result collection.

The following tracing function deliberately has a side effect so you can see when calls
happen. It is a teaching aid, not a pure transformation.


In [ ]:
calls = []
def traced_square(number):
    calls.append(number)
    return number * number

squares = map(traced_square, [1, 2, 3])
print("After construction:", calls)
print("First result:", next(squares), "calls:", calls)
print("Remaining:", list(squares), "calls:", calls)
print("After exhaustion:", list(squares))


## 7. Lists, iterators, and practical tradeoffs

A list comprehension eagerly computes and stores all results; it can be indexed, measured
with `len`, and iterated repeatedly. A `map`, `filter`, or generator expression produces
values on demand and usually supports only one pass. It may still hold references to its
input, so lazy does not mean "uses no memory."

Choose a list when you need repeated access or indexing, and a stream when you need one
pass. Performance depends on the operation and Python implementation; measure a real
workload before making speed claims. Empty input is a useful edge case for either approach.


In [ ]:
eager = [value * value for value in range(4)]
lazy = map(lambda value: value * value, range(4))
print(eager[2], len(eager), list(eager), list(eager))
print(list(lazy), list(lazy))
print(list(map(len, [])), list(filter(bool, [])))


## 8. Lambdas are small function expressions

`lambda parameters: expression` creates a function whose result is the expression's
value. It can accept multiple arguments, but its body is one expression, not a block of
statements. A lambda is useful for a short, single-use callback, especially a sorting key.

`def` also creates a function object and binds it to a name. Assigning a lambda to a name is
legal, but a normal `def` usually gives a reusable function a clearer name, docstring, and
traceback. A lambda is not automatically faster or purer than a function defined with `def`.


In [ ]:
pairs = [(4, 1), (3, -2), (8, 0)]
print(list(map(lambda value: value ** 2, range(5))))
print(list(filter(lambda pair: pair[1] > 0, pairs)))
print(sorted(pairs, key=lambda pair: pair[1]))
print((lambda x, y: x * y)(3, 4))
print((lambda value: value > 3)(4))

def triple(value):
    """Multiply a value by three."""
    return value * 3

print(triple(7))


## 9. Iterable versus iterator

An **iterable** can supply an iterator through `iter(value)`. Lists, strings, dictionaries,
and ranges are common iterables. An **iterator** has a current position: `next(iterator)`
returns the next value or raises `StopIteration` when no values remain. Its `__iter__`
returns itself. Every iterator is iterable, but a list is not itself an iterator.

Calling `iter` twice on a list makes two independent traversals. Calling `iter` on an
iterator gives that same iterator, not a reset copy. `range` is a reusable iterable, not a
single-use iterator; it also represents its values without storing them all in a list.


In [ ]:
values = [10, 20, 30]
left = iter(values)
right = iter(values)
print(next(left), next(left), next(right))
print(iter(left) is left)
print(list(range(3)), list(range(3)))
try:
    next(values)
except TypeError:
    print("A list supplies an iterator; it is not itself an iterator.")


## 10. Exhaustion is normal, and consumption changes the remaining stream

An exhausted iterator continues to raise `StopIteration`. A `for` loop handles this signal
for you. When taking individual values, `next(iterator, default)` can provide a fallback
instead of raising. Use a unique sentinel object when every ordinary value, including
`None`, might be valid data.

An iterator does not generally promise a snapshot of its input. A list iterator can observe
certain changes to the list; changing a dictionary's size during iteration typically raises
`RuntimeError`. Avoid structural mutations during traversal unless the behavior is deliberate
and documented. Iterate over a copy when you need an independent snapshot.


In [ ]:
iterator = iter([1, 2])
print(next(iterator), next(iterator))
try:
    next(iterator)
except StopIteration:
    print("The iterator is exhausted.")
sentinel = object()
print(next(iterator, sentinel) is sentinel)

data = ["a", "b"]
snapshot = iter(data.copy())
data.append("c")
print(list(snapshot), data)


## 11. How a `for` loop uses the protocol

Conceptually, a `for` loop obtains an iterator once, repeatedly calls `next`, and stops on
`StopIteration`. The manual version below catches exhaustion only around `next`. Exceptions
in the processing code should not be mistaken for the normal end of the input.

Most of the time the `for` version is clearer. Knowing the protocol explains why the same
loop works with lists, files, dictionary views, and generators without knowing how those
objects store or produce their values.


In [ ]:
letters = ["a", "b", "c"]
automatic = []
for letter in letters:
    automatic.append(letter.upper())

manual = []
iterator = iter(letters)
while True:  # Bounded by exhaustion of a finite three-element list.
    try:
        letter = next(iterator)
    except StopIteration:
        break
    manual.append(letter.upper())
print(automatic, manual)


## 12. Built-ins consume iterables in different ways

`sum`, `min`, `max`, and `list` need all values from a general finite stream. Membership,
`any`, and `all` can stop as soon as the answer is known. Values already requested remain
consumed, even when the consumer stops early. `enumerate`, `zip`, `map`, and `filter` return
iterators. `zip` normally ends at the shortest input; use `strict=True` when unequal finite
lengths would be a mistake.

Do not ask `list`, `sum`, or `max` to consume an unbounded stream. Even membership, `any`, or
`all` may run forever if the answer is never established. `all([])` is `True` and `any([])`
is `False`; these identities make them useful in validation pipelines.


In [ ]:
flags = iter([False, True, False, True])
print(any(flags))
print(list(flags))  # The first two values have already been consumed.
print(all([]), any([]))
print(min([3, 1, 4]), max([3, 1, 4]), sum([3, 1, 4]))
print(list(enumerate(["Ada", "Lin"], start=1)))
print(list(zip(["a", "b"], [10])))
try:
    list(zip(["a", "b"], [10], strict=True))
except ValueError:
    print("Strict zip detected unequal lengths.")


## 13. Generator expressions provide lazy comprehensions

Replace the square brackets of a list comprehension with parentheses to make a generator
expression: `(transform(value) for value in iterable)`. It is a single-use iterator. The
element expression and filters run as values are requested; the outermost iterable
expression is evaluated when the generator expression is created.

A generator expression works well as the input to an aggregate such as `sum`. If it is the
only argument, the function-call parentheses are enough. The input still needs to be finite
for a full aggregate to finish.


In [ ]:
squared = (number * number for number in range(6))
print(next(squared), list(squared), list(squared))
print(sum(number * number for number in range(6)))
print(list(number for number in range(10) if number % 3 == 0))


## 14. Laziness can avoid work when a consumer stops early

To test membership in a list comprehension, Python first builds the entire list. In a
generator expression, membership can stop when the target is found. The tracing lists below
show that the lazy version only transforms the needed inputs. If no input produces the
target, both versions must inspect the entire finite input.

This is a reason to keep transformations free of necessary side effects: skipped work in a
lazy pipeline must not accidentally skip something the program needed to do.


In [ ]:
visited = []
def tracked_double(value):
    visited.append(value)
    return value * 2

found_eager = 4 in [tracked_double(value) for value in range(6)]
print(found_eager, visited)
visited.clear()
found_lazy = 4 in (tracked_double(value) for value in range(6))
print(found_lazy, visited)
assert visited == [0, 1, 2]


## 15. A generator function pauses at `yield`

A function containing `yield` is a generator function. Calling it returns a generator
object; it does not start executing the body. `next` starts or resumes execution until the
next `yield`. Local variables and the execution position survive the pause. Reaching the
end, or using `return`, ends the stream and causes `StopIteration` for the consumer.

A normal function returns a result and does not resume from that return point. Its local
objects can still survive if referenced elsewhere, for example by a returned closure.
`yield from iterable` is a compact way to yield all the values of another iterable.


In [ ]:
def generate_ints(stop):
    for number in range(stop):
        yield number

generated = generate_ints(3)
print(type(generated).__name__)
print(next(generated), next(generated), next(generated))
print(next(generated, "finished"))
print(list(generate_ints(0)))

def joined_values():
    yield from [1, 2]
    yield from range(3, 5)

print(list(joined_values()))


## 16. The same protocol can be implemented with a class

This `Countdown` stores its current position in `remaining`. `__iter__` returns the iterator
itself; `__next__` updates the state or raises `StopIteration`. `list(Countdown(3))` consumes
the iterator using exactly those methods. A generator usually needs less code for a stream,
but the explicit class shows that iterators are ordinary objects with a protocol.

A zero countdown is empty. Once this iterator is exhausted, calling `list` again cannot
restart it; create a new `Countdown` for a new traversal.


In [ ]:
class Countdown:
    """A single-use iterator yielding start, start - 1, ..., 1."""

    def __init__(self, start):
        if isinstance(start, bool) or not isinstance(start, int):
            raise TypeError("start must be an integer")
        if start < 0:
            raise ValueError("start must not be negative")
        self.remaining = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.remaining == 0:
            raise StopIteration
        result = self.remaining
        self.remaining -= 1
        return result

countdown = Countdown(3)
print(iter(countdown) is countdown)
print(list(countdown), list(countdown), list(Countdown(0)))


## 17. Infinite streams need bounded consumers

An infinite Fibonacci generator can describe `1, 1, 2, 3, 5, ...` without building an
infinite collection. Its loop is intentional: the **consumer** decides how much to ask for.
`itertools.islice(stream, count)` takes at most `count` values and then stops. Constructing
the generator is safe; `list(fibonacci_numbers())` or `max(fibonacci_numbers())` would never
finish and must not be run.

This convention begins at `F(1)=1`; the indexed Fibonacci function later also defines
`F(0)=0`. Keeping the convention explicit prevents off-by-one misunderstandings.


In [ ]:
from itertools import islice

def fibonacci_numbers():
    """Yield the infinite stream 1, 1, 2, 3, 5, ...; consume it with a bound."""
    a, b = 0, 1
    while True:
        a, b = b, a + b
        yield a

stream = fibonacci_numbers()
print(list(islice(stream, 10)))
print("Next after those ten:", next(stream))
print(list(islice(fibonacci_numbers(), 0)))


## 18. A value bound is different from a count bound

`islice(..., 10)` limits how many values are requested. `fibonacci_up_to(20)` instead emits
values no greater than 20, stopping at the first value beyond that threshold. This works
because the positive Fibonacci sequence eventually exceeds every finite integer bound.
The terminating value has been requested from the inner stream even though it is not yielded.

The function rejects non-integer limits, including infinite floating-point values. Validation
inside a generator runs on its first advancement, not on the call that creates it. A zero
or negative bound yields no positive Fibonacci numbers.


In [ ]:
def fibonacci_up_to(limit):
    """Yield Fibonacci numbers no greater than a finite integer limit."""
    if isinstance(limit, bool) or not isinstance(limit, int):
        raise TypeError("limit must be an integer")
    for number in fibonacci_numbers():
        if number > limit:
            return
        yield number

print(list(fibonacci_up_to(20)))
print(list(fibonacci_up_to(1)), list(fibonacci_up_to(0)))
try:
    list(fibonacci_up_to(float("inf")))
except TypeError as error:
    print(type(error).__name__, error)


## 19. Closures retain access to enclosing names

A **closure** is a function that retains access to variables from an enclosing function
scope after that outer call has returned. `make_divisibility_test` returns a predicate
remembering its own divisor. Each factory call creates a distinct enclosing scope, so a
test for three and a test for five keep different divisors.

This is another form of composition: configure a function once, then pass it to `filter`.
Zero is rejected when constructing the predicate, because divisibility by zero is undefined.
Returning the inner function without parentheses returns the callable rather than calling it.


In [ ]:
def make_divisibility_test(divisor):
    """Return a predicate retaining its own nonzero integer divisor."""
    if isinstance(divisor, bool) or not isinstance(divisor, int):
        raise TypeError("divisor must be an integer")
    if divisor == 0:
        raise ValueError("divisor must not be zero")

    def divisible(number):
        return number % divisor == 0

    return divisible

by_three = make_divisibility_test(3)
by_five = make_divisibility_test(5)
print(list(filter(by_three, range(10))))
print(by_three(10), by_five(10), make_divisibility_test(5)(10))
try:
    make_divisibility_test(0)
except ValueError as error:
    print(error)


## 20. Closures look up names; they do not automatically freeze values

A closure normally reads an enclosing variable when it is called. Creating several
functions in a loop can therefore make every function use the loop variable's final value.
Use a factory call to give each function its own enclosing value. A default argument can
also bind the current value at function creation when that is the intended behavior.

Closures can hold state. `nonlocal` lets an inner function rebind an enclosing variable;
that is useful for a counter but makes it stateful rather than pure. A closure is a
language mechanism, not a guarantee of functional purity.


In [ ]:
late_functions = [lambda value: value * factor for factor in range(1, 4)]
print([function(10) for function in late_functions])
bound_functions = [lambda value, factor=factor: value * factor for factor in range(1, 4)]
print([function(10) for function in bound_functions])

def make_counter():
    count = 0
    def next_count():
        nonlocal count
        count += 1
        return count
    return next_count

counter = make_counter()
print(counter(), counter())


## 21. A decorator receives a function and returns a replacement

A **decorator** transforms a function, commonly by wrapping a call with extra behavior.
`debug` receives the original function and returns `wrapper`. The wrapper prints the call
arguments and then returns the original function's result. `*args` and `**kwargs` preserve
the caller's positional and keyword arguments.

`functools.wraps` copies useful metadata such as the name and docstring and provides
`__wrapped__` for access to the original function. Without it, the decorated function would
look like a function named `wrapper`, which makes documentation and debugging less helpful.
Printing is an intentional side effect of this decorator.


In [ ]:
from functools import wraps

def debug(function):
    """Print call arguments while preserving metadata, results and errors."""
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"Calling {function.__name__}: {args!r}, {kwargs!r}")
        return function(*args, **kwargs)
    return wrapper

def total(a, b, multiplier=1):
    """Add two values, then apply a multiplier."""
    return (a + b) * multiplier

total = debug(total)
print("Result:", total(2, 3, multiplier=4))
print("Name:", total.__name__)
print("Docstring:", total.__doc__)


## 22. `@decorator` is decoration at definition time

Placing `@debug` immediately before a function definition applies `debug` and binds the
function's name to the result. For this example it is equivalent to defining the function
and then assigning `function = debug(function)`. Decoration happens when the definition
executes; the wrapper body runs later, each time the decorated function is called.

The decorator must return the replacement callable. The wrapper must return the wrapped
function's result if callers expect that result. Omitting either return is a common mistake
that turns a useful function or its result into `None`.


In [ ]:
@debug
def scaled_total(a, b, multiplier=1):
    """Add and scale two values."""
    return (a + b) * multiplier

result = scaled_total(5, 3, multiplier=2)
print(result, scaled_total.__name__)
print("Original result without tracing:", scaled_total.__wrapped__(5, 3, multiplier=2))
assert result == 16


## 23. Preserve calling behavior, including failures

A transparent wrapper forwards positional and keyword arguments, returns the result, and
lets the original exceptions propagate. Do not catch an exception just because a decorator
is present. The caller still needs to know that the operation failed.

This example has a keyword-only parameter, so it checks more than a simple positional call.
The outer `try` catches the expected failure only for the demonstration.


In [ ]:
@debug
def divide(numerator, *, denominator):
    """Divide using an explicit keyword denominator."""
    return numerator / denominator

assert divide(12, denominator=3) == 4
assert divide.__name__ == "divide"
try:
    divide(12, denominator=0)
except ZeroDivisionError:
    print("The wrapped failure reached the caller.")


## 24. Decorator factories configure the wrapper

A decorator with an argument needs one extra level: a factory receives the configuration
and returns the decorator, which receives the function and returns the wrapper.
`@scale_result(10)` first calls the factory with 10, then applies the resulting decorator.

Several decorators are applied from the bottom upward. For `@outer` above `@inner`, the
definition becomes `function = outer(inner(function))`. Think through that order when the
wrappers change values or add effects. Avoid clever stacks that hide the main operation.


In [ ]:
def scale_result(factor):
    """Build a decorator that multiplies a numeric return value."""
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            return factor * function(*args, **kwargs)
        return wrapper
    return decorate

@scale_result(10)
def total_score(a, b=0):
    """Return the combined score."""
    return a + b

print(total_score(2, b=3), total_score.__name__)

@debug
@scale_result(2)
def bonus_score(value):
    return value + 1

print(bonus_score(4))


## 25. Memoization with `functools`

**Memoization** stores a function result by its arguments and reuses it for the same call.
Recursive Fibonacci repeats many subproblems; caching means each index is calculated only
once for the small example below. `cache_info` shows hits, misses, and stored entries;
`cache_clear` resets them. `F(0)=0`, `F(1)=1`, and `F(n)=F(n-1)+F(n-2)`.

`@cache` is shorthand for `@lru_cache(maxsize=None)`. This example instead sets `typed=True`
so arguments of different types get separate cache entries. That matters because we promise
to reject booleans and floats, yet `1 == True == 1.0`. With untyped keys, a cached integer
keyword call could supply a result for an invalid type before the function body can check it.

Use caching for deterministic computations with hashable arguments. The cache retains
arguments and results, so unbounded variety can consume memory; `lru_cache(maxsize=...)`
provides a size limit. Avoid caching time-dependent values or required side effects, and be
careful about sharing a cached mutable result. Caching does not remove Python's recursion
limit: use an iterative algorithm for very large Fibonacci indices.


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None, typed=True)
def fibonacci_at(index):
    """Return F(index) for a small nonnegative integer; F(0)=0, F(1)=1.

    Typed keys keep booleans and floats separate from validated integer keys.
    Caching avoids repeated recursive subproblems. Very large indices should
    use an iterative algorithm to avoid Python's recursion limit.
    """
    if isinstance(index, bool) or not isinstance(index, int):
        raise TypeError("index must be an integer")
    if index < 0:
        raise ValueError("index must not be negative")
    if index < 2:
        return index
    return fibonacci_at(index - 1) + fibonacci_at(index - 2)

fibonacci_at.cache_clear()
print(fibonacci_at(30))
before = fibonacci_at.cache_info()
print(fibonacci_at(30))
after = fibonacci_at.cache_info()
print("Before repeat:", before)
print("After repeat:", after)
assert after.hits == before.hits + 1


## 26. Decorators are used beyond tracing

The same syntax appears in `@property` (an attribute interface), `@classmethod` (a method
receiving its class), and `@staticmethod` (no automatic instance or class argument).
Here a class method is a named factory and a property is read-only. They are standard
library features; Chapter 05 explains the surrounding object model in more detail.

Libraries also use decorators to register routes, check access, or arrange timing behavior.
A decorator only does what its implementation supplies: writing `@timeout` would not by
itself interrupt a blocking operation. Understand a library decorator's contract before using it.


In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        return cls((fahrenheit - 32) * 5 / 9)

    @staticmethod
    def unit():
        return "degrees Celsius"

    @property
    def celsius(self):
        return self._celsius

temperature = Temperature.from_fahrenheit(68)
print(temperature.celsius, Temperature.unit())


## Practice

For each exercise, test empty input and at least one case that would reveal a misleading
implementation. Check laziness by recording calls, and check decorator behavior with a
keyword argument and a returned value. The starter cells contain comments so the unfinished
notebook remains executable from a fresh kernel.


### Exercise 1 · A pure text-cleaning pipeline

Write `cleaned_words(words)` returning a lazy iterator that strips whitespace, lowercases
text, and drops empty results. Implement it with `map` and `filter`; then write a
comprehension producing the same values. Verify that the input list is unchanged, empty
input works, and consuming the returned iterator twice yields no values on its second pass.


In [ ]:
# TODO: Implement cleaned_words and compare it with a comprehension.
# Example: [" Python ", "", " DATA "] should produce ["python", "data"].


### Exercise 2 · A prime predicate and a sorting key

Write `is_prime(number)` for integer inputs. Values below two are not prime. Use it with
`filter` to select primes below 30 and compare with a comprehension. Test negative values,
0, 1, 2, a square such as 49, and a prime such as 97. Then sort
`[("Ada", 8), ("Lin", 5), ("Bo", 8)]` by descending score and alphabetical name using a
single lambda key; do not mutate the original list.


In [ ]:
# TODO: Implement is_prime and the two-field sorting key.
# Expected sorting order: [("Ada", 8), ("Bo", 8), ("Lin", 5)]


### Exercise 3 · A bounded Fibonacci stream

Write an infinite generator producing `1, 1, 2, 3, ...`, then `first_fibonacci(count)` that
returns a list of exactly `count` values using `islice`. Count must be a nonnegative integer,
excluding `bool`. Reject wrong types with `TypeError` and negative counts with `ValueError`.
Test counts 0, 1, and 8, and show that two newly created generators advance independently.
Never convert the unbounded generator directly to a list.


In [ ]:
# TODO: Import islice, define the generator and bounded consumer, and test limits.
# Eight values: [1, 1, 2, 3, 5, 8, 13, 21]


### Exercise 4 · Configured predicates with closures

Write `make_divisibility_test(divisor)` that returns a predicate. The divisor must be a
nonzero integer, excluding `bool`; use `TypeError` for the wrong type and `ValueError` for
zero. Build predicates for 3, 5, and -2. Check zero as a tested number, negative numbers,
and independent configuration. Use your predicate for 3 with `filter(range(10))` in the
correct calling form. Explain why returning the function differs from calling it.


In [ ]:
# TODO: Return a nested function that retains divisor, then use it with filter.
# Expected multiples of 3 in range(10): [0, 3, 6, 9]


### Exercise 5 · A decorator that counts calls

Write `count_calls(function)` using `functools.wraps`. Its wrapper must forward all
arguments, return the original result, and expose `wrapper.calls`, initially zero. Increment
the count before each attempted call, including calls that raise. Test a function with a
keyword-only argument, its name and docstring, a failed call, and two independently
decorated functions. The decorator should let the original exception propagate.


In [ ]:
# TODO: Implement count_calls and verify metadata, return values, and errors.
# Use @count_calls on a small function that can deliberately raise ValueError.


### Exercise 6 · Cache repeated work

Write a Fibonacci function decorated with `@lru_cache(maxsize=None, typed=True)` for small
nonnegative integer indices, with `F(0)=0` and `F(1)=1`. Reject booleans and non-integers with
`TypeError`, and negative indices with
`ValueError`. Clear the cache, calculate `F(20)`, record `cache_info()`, and call `F(20)` again.
Assert that the result is unchanged, misses do not increase, and hits increase by one.
Also cache `index=1`, then check that keyword calls with `index=True` and `index=1.0` still fail.
Explain why a function that returns the current time is a poor candidate for this cache.


In [ ]:
# TODO: Import lru_cache, implement the function, and compare cache_info before/after.
# Expected F(20): 6765


## Check your understanding and continue

You should now be able to choose between a reusable collection and a single-pass stream,
identify when a lazy transformation runs, bound an infinite generator, and preserve a
function's arguments and result through a decorator. A useful design pattern is to keep
the core transformations pure and put input/output around them.

Official references: [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html),
[iterator types](https://docs.python.org/3/library/stdtypes.html#iterator-types),
[itertools](https://docs.python.org/3/library/itertools.html), and
[functools](https://docs.python.org/3/library/functools.html).

This chapter adapts `6_FP.pdf`, pages 1–50. It clarifies Python 3 laziness, iterator exhaustion,
closure capture, and decorator metadata while keeping every example finite to execute.
